# Customer Churn Prediction — Exploratory Data Analysis

This notebook explores the Telco Customer Churn dataset and visualizes key patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
from data_preprocessing import load_data

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

df = load_data()
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Churn distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df['Churn'].value_counts().plot.pie(autopct='%1.1f%%', ax=axes[0], colors=['#2ecc71', '#e74c3c'])
axes[0].set_title('Churn Distribution')
axes[0].set_ylabel('')

sns.countplot(data=df, x='Churn', hue='Contract', ax=axes[1])
axes[1].set_title('Churn by Contract Type')
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by tenure groups
df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72], labels=['0-12', '13-24', '25-48', '49-72'])

churn_by_tenure = df.groupby('tenure_group')['Churn'].apply(lambda x: (x == 'Yes').mean())
churn_by_tenure.plot(kind='bar', color='#e74c3c', edgecolor='black')
plt.title('Churn Rate by Tenure Group (months)')
plt.ylabel('Churn Rate')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Monthly charges distribution by churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, color in [('Yes', '#e74c3c'), ('No', '#2ecc71')]:
    subset = df[df['Churn'] == label]
    axes[0].hist(subset['MonthlyCharges'], bins=30, alpha=0.6, label=f'Churn={label}', color=color)
axes[0].set_title('Monthly Charges by Churn')
axes[0].legend()

for label, color in [('Yes', '#e74c3c'), ('No', '#2ecc71')]:
    subset = df[df['Churn'] == label]
    axes[1].hist(subset['tenure'], bins=30, alpha=0.6, label=f'Churn={label}', color=color)
axes[1].set_title('Tenure by Churn')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by key categorical features
cat_features = ['InternetService', 'PaymentMethod', 'OnlineSecurity', 'TechSupport']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feat in zip(axes.flatten(), cat_features):
    churn_rate = df.groupby(feat)['Churn'].apply(lambda x: (x == 'Yes').mean()).sort_values()
    churn_rate.plot(kind='barh', ax=ax, color='#3498db', edgecolor='black')
    ax.set_title(f'Churn Rate by {feat}')
    ax.set_xlabel('Churn Rate')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numeric features
from data_preprocessing import encode_features

df_encoded = encode_features(df.drop(columns=['tenure_group']))
corr = df_encoded.corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, annot=False, fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

# Top correlations with Churn
print('Top features correlated with Churn:')
print(corr['Churn'].drop('Churn').abs().sort_values(ascending=False).head(10))